# Machine Learning Zoomcamp 2026
## Homework 3: Machine Learning for Classification

This notebook provides the complete solution for Homework 3.


### 1. Imports and Data Loading

We download and load the 2026 course lead scoring dataset.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score, accuracy_score

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
try:
    df = pd.read_csv('course_lead_scoring_2026.csv')
except Exception:
    df = pd.read_csv(url)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


Dataset shape: (5000, 9)
Columns: ['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']


### 2. Data Preparation

- Missing values in categorical features are replaced with `'NA'`.
- Missing values in numerical features are replaced with `0.0`.

In [2]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

print("Missing values before preparation:")
print(df.isnull().sum())

for col in categorical:
    df[col] = df[col].fillna('NA')

for col in numerical:
    df[col] = df[col].fillna(0.0)

print("\nMissing values after preparation:")
print(df.isnull().sum())


Missing values before preparation:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

Missing values after preparation:
lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


### Question 1
**What is the most frequent observation (mode) for the column `industry`?**

- `NA`
- **`technology`** (Correct)
- `healthcare`
- `retail`

In [3]:
print("Value counts for industry:")
print(df['industry'].value_counts())
print(f"\nMost frequent observation (mode): {df['industry'].mode()[0]}")


Value counts for industry:
industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

Most frequent observation (mode): technology


### Question 2
**Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.**

**What are the two features that have the biggest correlation?**

- **`interaction_count` and `lead_score`** (Correct)
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

In [4]:
corr_matrix = df[numerical].corr()
print("Correlation matrix:")
print(corr_matrix)

q2_pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

print("\nCorrelations for candidate pairs:")
for f1, f2 in q2_pairs:
    print(f"  {f1} and {f2}: {corr_matrix.loc[f1, f2]:.5f}")


Correlation matrix:
                          annual_income  number_of_courses_viewed  interaction_count  lead_score
annual_income                  1.000000                  0.161300           0.122842    0.229496
number_of_courses_viewed       0.161300                  1.000000           0.721611    0.757204
interaction_count              0.122842                  0.721611           1.000000    0.915746
lead_score                     0.229496                  0.757204           0.915746    1.000000

Correlations for candidate pairs:
  interaction_count and lead_score: 0.91575
  number_of_courses_viewed and lead_score: 0.75720
  number_of_courses_viewed and interaction_count: 0.72161
  annual_income and interaction_count: 0.12284

Pair with biggest correlation: interaction_count and lead_score (0.91575)


### Split the data

Split the data using the exact specified calls:
```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)
```

In [5]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

print(f"Training set size: {len(df_train)}")
print(f"Validation set size: {len(df_val)}")
print(f"Test set size: {len(df_test)}")


Training set size: 3000
Validation set size: 1000
Test set size: 1000


### Question 3
**Calculate the mutual information score between `converted` and other categorical variables in the dataset. Use the training set only. Round the scores to 2 decimals using `round(score, 2)`.**

**Which of these variables has the biggest mutual information score?**

- `industry`
- `location`
- **`lead_source`** (Correct)
- `employment_status`

In [6]:
mi_scores = {}
print("Mutual Information scores:")
for col in categorical:
    mi = mutual_info_score(df_train[col], y_train)
    mi_scores[col] = mi
    print(f"  {col}: raw={mi:.6f}, round(2)={round(mi, 2):.2f}")

biggest_mi = max(mi_scores, key=mi_scores.get)
print(f"\nVariable with biggest mutual information: {biggest_mi}")


Mutual Information scores:
  lead_source: raw=0.034346, round(2)=0.03
  industry: raw=0.001995, round(2)=0.00
  employment_status: raw=0.018966, round(2)=0.02
  location: raw=0.001043, round(2)=0.00

Variable with biggest mutual information: lead_source


### Question 4
**Now let's train a logistic regression.**
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset with parameters: `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`.
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

**What accuracy did you get?**

- 0.55
- **0.65** (Correct)
- 0.75
- 0.85

In [7]:
dv = DictVectorizer(sparse=False)
train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
acc_orig = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: raw={acc_orig:.6f}, round(2)={round(acc_orig, 2):.2f}")


Validation Accuracy: raw=0.645000, round(2)=0.65


### Question 5
**Let's find the least useful feature using the *feature elimination* technique.**
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

**Which of following feature has the smallest difference?**

- `'lead_source'`
- **`'number_of_courses_viewed'`** (Correct)
- `'interaction_count'`

In [8]:
all_features = categorical + numerical
diffs = {}
q5_candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

print(f"Original accuracy: {acc_orig:.6f}\n")
print("Feature elimination results:")
for feature in all_features:
    subset = [f for f in all_features if f != feature]
    dv_sub = DictVectorizer(sparse=False)
    X_train_sub = dv_sub.fit_transform(df_train[subset].to_dict(orient='records'))
    X_val_sub = dv_sub.transform(df_val[subset].to_dict(orient='records'))

    model_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_sub.fit(X_train_sub, y_train)
    acc_sub = accuracy_score(y_val, model_sub.predict(X_val_sub))
    diff = acc_orig - acc_sub
    diffs[feature] = diff
    print(f"  Without '{feature}': acc = {acc_sub:.6f}, diff = {diff:.6f}")

print("\nCandidate options:")
for c in q5_candidates:
    print(f"  '{c}': diff = {diffs[c]:.6f}")

best_q5 = min(q5_candidates, key=lambda c: diffs[c])
print(f"\nSmallest difference among options: {best_q5} (diff = {diffs[best_q5]:.6f})")


Original accuracy: 0.645000

Feature elimination results:
  Without 'lead_source': acc = 0.642000, diff = 0.003000
  Without 'industry': acc = 0.645000, diff = 0.000000
  Without 'employment_status': acc = 0.645000, diff = 0.000000
  Without 'location': acc = 0.645000, diff = 0.000000
  Without 'annual_income': acc = 0.724000, diff = -0.079000
  Without 'number_of_courses_viewed': acc = 0.643000, diff = 0.002000
  Without 'interaction_count': acc = 0.601000, diff = 0.044000
  Without 'lead_score': acc = 0.644000, diff = 0.001000

Candidate options:
  'lead_source': diff = 0.003000
  'number_of_courses_viewed': diff = 0.002000
  'interaction_count': diff = 0.044000

Smallest difference among options: number_of_courses_viewed (diff = 0.002000)


### Question 6
**Now let's train a regularized logistic regression.**
- Let's try the following values of the parameter `C`: `[0.000001, 0.00001, 0.0001, 0.001]`.
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

**Which of these `C` leads to the best accuracy on the validation set?**

- `0.000001`
- `0.00001`
- `0.0001`
- **`0.001`** (Correct)

In [9]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]
c_accs = {}
print("Tuning parameter C:")
for c_val in c_values:
    m_c = LogisticRegression(solver='liblinear', C=c_val, max_iter=1000, random_state=42)
    m_c.fit(X_train, y_train)
    acc_c = accuracy_score(y_val, m_c.predict(X_val))
    rounded_acc = round(acc_c, 3)
    c_accs[c_val] = (acc_c, rounded_acc)
    print(f"  C = {c_val}: accuracy = {acc_c:.6f}, round(3) = {rounded_acc:.3f}")

max_acc = max(c_accs[c][1] for c in c_values)
best_c_candidates = [c for c in c_values if c_accs[c][1] == max_acc]
best_c = min(best_c_candidates)
print(f"\nBest accuracy is {max_acc} achieved at C = {best_c}")


Tuning parameter C:
  C = 1e-06: accuracy = 0.598000, round(3) = 0.598
  C = 1e-05: accuracy = 0.598000, round(3) = 0.598
  C = 0.0001: accuracy = 0.613000, round(3) = 0.613
  C = 0.001: accuracy = 0.645000, round(3) = 0.645

Best accuracy is 0.645 achieved at C = 0.001


### Summary of Answers

| Question | Question Description | Selected Answer |
| :--- | :--- | :--- |
| **Question 1** | Mode for `industry` | **`technology`** |
| **Question 2** | Two features with biggest correlation | **`interaction_count` and `lead_score`** |
| **Question 3** | Categorical feature with biggest mutual information score | **`lead_source`** |
| **Question 4** | Logistic regression validation accuracy | **`0.65`** |
| **Question 5** | Feature with smallest difference in accuracy upon elimination | **`number_of_courses_viewed`** |
| **Question 6** | Value of `C` leading to best validation accuracy | **`0.001`** |
